In [1]:
import pandas as pd
import os
import deepl
from dotenv import load_dotenv
from deep_translator import GoogleTranslator

In [3]:
# Load environment variables from .env file
load_dotenv()
DEEPL_API_KEY = os.getenv("DEEPL_API_KEY") 
auth_key = DEEPL_API_KEY # replace with your key
deepl_client = deepl.DeepLClient(auth_key)

In [ ]:
# === Input file path ===
input_path = r"C:\Users\SakshiMehta\OneDrive - Denison Consulting\Text Analytics data\Mobily\Mobily OCS 202606_SurveyData_2026-07-07_0854.xlsx"
company_name = "Mobily"
EU_client = False

In [5]:
df = pd.read_excel(input_path, keep_default_na=False) 

#  Get folder path of input file 
output_folder = os.path.dirname(input_path)
print(output_folder)

C:\Users\SakshiMehta\OneDrive - Denison Consulting\Text Analytics data\Real Madrid


In [6]:
def preprocess_data(df, cols_to_drop=None):

    df.columns = df.iloc[0] # Set the first row as the header
    df = df[2:] # Skip the first two rows which are not part of the data
    df.reset_index(drop=True, inplace=True)


    df = df.iloc[:, 4:]  # Drop the first 4 column
    # Keep column "Status"
    status_idx = df.columns.get_loc("Status")
 
    # Find the first column starting with "St" after "Status"
    after_status = df.columns[status_idx + 1:]
  
    oe_cols = [col for col in after_status if col.startswith("St")]
    if oe_cols:
        first_oe_idx = df.columns.get_loc(oe_cols[0])
        # Drop columns between "Status" and the first "Oe"
        df = df.drop(columns=df.columns[status_idx + 1:first_oe_idx])

        # Drop extra columns if provided
    if cols_to_drop:
        df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    # Filter rows where Status is "Submitted - Valid"
    df = df[df["Status"] == "Submitted - Valid"].reset_index(drop=True)

    df = df.dropna(axis=1, how='all')
    # Drop columns where all values are either NaN or empty/whitespace strings
    df = df.loc[:, ~df.apply(lambda col: col.astype(str).str.strip().eq('').all())]



    return df

df_cleaned = preprocess_data(df)

In [ ]:
oe_columns = [col for col in df_cleaned.columns if col.startswith("St") and col != "Status" and col != "store"]
oe_dfs = {}  # dictionary to store separate dataframes

# Initialize translator
if not EU_client:
    # Initialize translator
    translator = GoogleTranslator(source='auto', target='en')


for oe_col in oe_columns:
    # Keep all non-'oe' columns + the current 'oe' column
    cols_to_keep = [col for col in df_cleaned.columns if not col in oe_columns or col == oe_col]
    
    # Create the filtered DataFrame
    df_part = df_cleaned[cols_to_keep].copy()
    # Remove rows where the current oe_col is blank or only whitespace
    df_part = df_part[df_part[oe_col].astype(str).str.strip() != '']
    # Add an index column named "Res_ID" as the first column
    df_part.insert(0, 'Res_ID', range(1, len(df_part) + 1))

  
    # Translate the 'oe_col' column to English
    translated_col = f"{oe_col}_Trans"
 
    if EU_client:
        df_part[translated_col] = df_part[oe_col].apply(lambda x: deepl_client.translate_text(str(x), target_lang="EN-US").text)

    else:
        df_part[translated_col] = df_part[oe_col].apply(lambda x: translator.translate(str(x)))

    # Move the translated column to the third position
    df_part.insert(2, translated_col, df_part.pop(translated_col))

    # replace all the blank values in the translated column with the respective original values
    df_part[translated_col] = df_part.apply(
    lambda row: row[oe_col] if pd.isna(row[translated_col]) or str(row[translated_col]).strip() == '' else row[translated_col],
    axis=1
)
    # Sort by length of translated text (descending)
    df_part = df_part.sort_values(by=translated_col, key=lambda col: col.str.len(), ascending=True)

    # Save to dictionary
    oe_dfs[oe_col] = df_part
    
    translator_name = "DeepL" if EU_client else "Google"
    output_path = os.path.join(output_folder, f"{company_name}_{oe_col}_{translator_name}_Trans.xlsx"
)
    df_part.to_excel(output_path, index=False)
